# Sky Foreground on the Corner Wavefront Sensors — Post-ISR Images (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-06-12
**Last Modified:** 2026-06-12
**Status:** In Progress
**Keywords:** WFS, corner wavefront sensor, post-ISR, sky background, AOS, FAM

## Description

Start of a study of the **sky foreground shape** on the Rubin LSST Camera corner
wavefront sensors (CWFS). This notebook locates ISR-processed images for the eight
corner-WFS half-chips and makes PNG plots with a useful stretch.

Key functionality:
1. Query the AOS `cwfs` collection for `post_isr_image` datasets (the ISR output).
2. List the available exposures / nights and pick one to inspect.
3. Plot all eight corner-WFS half-chips for a chosen exposure, and save PNGs.

**Output:** PNG plots of corner-WFS post-ISR images in `wfs/output/`.

**Based on:** AOS `cwfs` reductions in
`LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2` (repo `/repo/main`).

> **Note on dataset type.** The original goal was `preliminary_visit_image` (PVI),
> but PVI is a full-focal-plane / FAM data product and is **not** produced by the
> corner-WFS-only `cwfs` pipeline. The ISR output that exists for the corner sensors
> is `post_isr_image`, which is the right product for studying sky-background shape
> (ISR-corrected, sky background still present). We use `post_isr_image` here.
>
> The exposures in this collection are tagged `infocus_aos_stability_test`, i.e.
> **in-focus** images on the corner sensors — exactly the in-focus FAM-triplet data
> of interest.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-12 | Aaron Roodman | Initial version: locate and visualize corner-WFS post-ISR images |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Inspect Available Exposures](#exposures)
6. [Visualize Corner-WFS Images](#visualize)
7. [Save PNGs](#save)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

butler_repo = "/repo/main"
collection = "LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2"
dataset_type = "post_isr_image"
instrument = "LSSTCam"

# Night to inspect (day_obs). Available in this collection:
#   20260315, 20260316, 20260317, 20260319, 20260324, 20260325, 20260326,
#   20260327, 20260331, 20260401, 20260402, 20260404, 20260409, 20260418,
#   20260419, 20260420, 20260423, 20260428
day_obs = 20260401

# Exposure to plot. None -> use the first exposure found on `day_obs`.
exposure = None

# The eight corner wavefront-sensor half-chips (detector id -> raft/sensor name).
WFS_DETECTORS = {
    191: "R00_SW0", 192: "R00_SW1",   # R00 raft corner
    195: "R04_SW0", 196: "R04_SW1",   # R04 raft corner
    199: "R40_SW0", 200: "R40_SW1",   # R40 raft corner
    203: "R44_SW0", 204: "R44_SW1",   # R44 raft corner
}

# ---- Display stretch ----------------------------------------------------
# These corner-WFS frames are sky-dominated (~6700 ADU) with a small dynamic
# range above sky, so the stretch matters. Modes:
#   "sky"        : sigma-clipped, centered on the sky level (best for sky shape)
#   "zscale"     : astropy ZScaleInterval (good general purpose)
#   "percentile" : low/high percentile clip
stretch_mode = "sky"

# "sky" mode: limits = [median - lo*sigma, median + hi*sigma]
sky_lo_sigma = 2.0
sky_hi_sigma = 8.0

# "zscale" mode
zscale_contrast = 0.25
zscale_nsamples = 100_000        # large arrays need many samples or limits collapse

# "percentile" mode
pct_low, pct_high = 1.0, 99.0

asinh_a = 0.1                    # AsinhStretch softening (applies to all modes)
cmap = "gray"

# Output
output_dir = "output"
save_pngs = True
dpi = 150


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.stats import sigma_clipped_stats
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize

import lsst.daf.butler as dafButler

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

os.makedirs(output_dir, exist_ok=True)


<a id='functions'></a>
## Helper Functions

In [ ]:
def list_exposures(butler, collection, dataset_type, instrument, day_obs):
    """Return sorted exposures available for a night, with key metadata."""
    refs = list(butler.query_datasets(
        dataset_type, collections=collection,
        where=f"instrument=\'{instrument}\' and exposure.day_obs={day_obs}",
        limit=None,
    ))
    by_exp = {}
    for r in refs:
        by_exp.setdefault(r.dataId["exposure"], set()).add(r.dataId["detector"])

    rows = []
    for exp in sorted(by_exp):
        (rec,) = butler.registry.queryDimensionRecords(
            "exposure", where=f"instrument=\'{instrument}\' and exposure={exp}")
        rows.append(dict(
            id=exp, n_detectors=len(by_exp[exp]),
            obs_reason=rec.observation_reason, physical_filter=rec.physical_filter,
            exptime=rec.exposure_time,
        ))
    return rows


def load_wfs_image(butler, collection, dataset_type, instrument, exposure, detector):
    """Load one post-ISR corner-WFS image; return the 2D float array."""
    exp = butler.get(dataset_type, collections=collection,
                     dataId={"instrument": instrument, "exposure": exposure,
                             "detector": detector})
    return np.asarray(exp.image.array, dtype=float)


def make_norm(data, mode="sky", *, sky_lo_sigma=2.0, sky_hi_sigma=8.0,
              zscale_contrast=0.25, zscale_nsamples=100_000,
              pct_low=1.0, pct_high=99.0, asinh_a=0.1):
    """Build an ImageNormalize (asinh stretch) for a sky-dominated frame.

    mode : "sky" | "zscale" | "percentile"
    """
    finite = data[np.isfinite(data)]
    if finite.size == 0:
        vmin, vmax = 0.0, 1.0
    elif mode == "sky":
        _, med, std = sigma_clipped_stats(finite, sigma=3.0, maxiters=5)
        vmin, vmax = med - sky_lo_sigma * std, med + sky_hi_sigma * std
    elif mode == "zscale":
        vmin, vmax = ZScaleInterval(contrast=zscale_contrast,
                                    n_samples=zscale_nsamples).get_limits(finite)
    elif mode == "percentile":
        vmin, vmax = np.nanpercentile(finite, [pct_low, pct_high])
    else:
        raise ValueError(f"unknown stretch mode {mode!r}")
    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + 1.0
    return ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch(a=asinh_a))


def plot_corner_wfs(butler, collection, dataset_type, instrument, exposure,
                    wfs_detectors, *, mode="sky", cmap="gray", norm_kwargs=None):
    """Plot all corner-WFS half-chips for one exposure in a 4x2 grid.

    Rows are raft corners (R00, R04, R40, R44); columns are SW0 / SW1.
    Returns the matplotlib Figure.
    """
    norm_kwargs = norm_kwargs or {}
    rafts = sorted({name.split("_")[0] for name in wfs_detectors.values()})
    fig, axes = plt.subplots(len(rafts), 2, figsize=(9, 4.0 * len(rafts)))
    axes = np.atleast_2d(axes)
    name_to_det = {v: k for k, v in wfs_detectors.items()}

    for i, raft in enumerate(rafts):
        for j, sw in enumerate(("SW0", "SW1")):
            ax = axes[i, j]
            name = f"{raft}_{sw}"
            det = name_to_det.get(name)
            if det is None:
                ax.axis("off")
                continue
            try:
                data = load_wfs_image(butler, collection, dataset_type,
                                      instrument, exposure, det)
                norm = make_norm(data, mode=mode, **norm_kwargs)
                ax.imshow(data, origin="lower", cmap=cmap, norm=norm,
                          aspect="equal", interpolation="nearest")
                med = np.nanmedian(data)
                ax.set_title(f"{name}  (det {det})  median={med:.1f}", fontsize=10)
            except Exception as e:
                ax.text(0.5, 0.5, f"{name}\n{type(e).__name__}", ha="center",
                        va="center", transform=ax.transAxes, fontsize=9)
                ax.set_title(name, fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle(f"Corner-WFS post-ISR — exposure {exposure}  (stretch: {mode})",
                 fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.99])
    return fig


<a id='data'></a>
## Data Access

In [ ]:
butler = dafButler.Butler(butler_repo, collections=collection)
print("Repo:", butler_repo)
print("Collection:", collection)
print("Dataset type:", dataset_type)


<a id='exposures'></a>
## Inspect Available Exposures

In [ ]:
rows = list_exposures(butler, collection, dataset_type, instrument, day_obs)
print(f"day_obs {day_obs}: {len(rows)} exposures with {dataset_type}\n")
print(f"{'exposure':>14} {'#det':>5} {'filter':>10} {'exptime':>8}  obs_reason")
for r in rows:
    print(f"{r['id']:>14} {r['n_detectors']:>5} {str(r['physical_filter']):>10} "
          f"{r['exptime']:>8.1f}  {r['obs_reason']}")

if exposure is None and rows:
    exposure = rows[0]["id"]
print("\nUsing exposure:", exposure)


<a id='visualize'></a>
## Visualize Corner-WFS Images

In [ ]:
norm_kwargs = dict(
    sky_lo_sigma=sky_lo_sigma, sky_hi_sigma=sky_hi_sigma,
    zscale_contrast=zscale_contrast, zscale_nsamples=zscale_nsamples,
    pct_low=pct_low, pct_high=pct_high, asinh_a=asinh_a,
)
fig = plot_corner_wfs(butler, collection, dataset_type, instrument, exposure,
                      WFS_DETECTORS, mode=stretch_mode, cmap=cmap,
                      norm_kwargs=norm_kwargs)
plt.show()


<a id='save'></a>
## Save PNGs

In [ ]:
if save_pngs:
    out = Path(output_dir) / f"wfs_corner_postisr_{exposure}_{stretch_mode}.png"
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print("Saved:", out)
